# 标普 500 收盘点位预测

固定截止日预测未来 5 或 20 个交易日，比较 Naive、XGBoost 和 LSTM。Notebook 与独立脚本共用模型、窗口和评估实现，核心模型源码在第二部分直接展示。本机选择 Python (SP500 Notebook) 内核，从上到下运行。20 日评估需逐窗口训练，耗时最长。

## 1. 读取数据

特征表提供模型输入。清洗表用于核对评估期的日期、收盘点位和切分标记。按日期检查训练、验证、测试三段的记录数和起止日。

In [1]:
from pathlib import Path
import inspect
import sys

import pandas as pd
from IPython.display import Code, display

cwd = Path.cwd()
roots = (cwd, *cwd.parents, cwd / "sp500-multitask-main")
folder = next((root / "data/processed" for root in roots
               if (root / "data/processed/sp500_features.csv").is_file()), None)
if folder is None:
    raise FileNotFoundError("请从项目根目录或 Notebook 所在目录运行。")
sys.path.insert(0, str(folder.parents[1] / "src/1_TimeSerialAnalyze"))

import ForecastModels as models
from EvaluateForecastMetrics import load_data, evaluate, metrics

data, clean = load_data(folder)
train = data[data.segment == "train"]
display(data.groupby("segment", sort=False).Date.agg(["size", "min", "max"]))


,size,min,max
segment,,,
train,5083,1997-08-29,2017-11-08
val,1132,2017-11-09,2022-05-10
test,1132,2022-05-11,2026-11-13


## 2. 训练模型

Naive 将截止日收盘点位沿用到整个窗口。XGBoost 为每个预测步长分别学习点位变化。本格直接展示共享脚本中的核心代码，实际训练在评估时调用。

In [2]:
display(Code(inspect.getsource(models.predict_xgboost), language="python"))

def predict_xgboost(train, cutoffs, horizon=SHORT_HORIZON):
    """Fit one model per step and return closing prices at all cutoffs."""
    forecasts = []
    for day in range(1, horizon + 1):
        rows = complete_cutoffs(train, horizon=day)
        if not len(rows):
            raise ValueError(f"No complete {day}-day training targets.")
        current = train.iloc[rows]
        target = train.Close.iloc[rows + day].to_numpy() - current.Close.to_numpy()
        model = XGBRegressor(
            objective="reg:absoluteerror",
            n_estimators=100, max_depth=2, learning_rate=0.03,
            tree_method="hist", random_state=42, n_jobs=2,
        )
        model.fit(current[FEATURES], target)
        forecasts.append(model.predict(cutoffs[FEATURES]))
    return cutoffs.Close.to_numpy()[:, None] + np.column_stack(forecasts)

LSTM 使用连续 30 个 NYSE 交易日的 10 个特征，一次输出整个窗口的点位变化。
下面展示共享脚本中的日期对齐、模型与训练代码；缺失日两侧不完整的训练样本会被剔除。

In [3]:
functions = (models.complete_cutoffs, models.ForecastModel, models.make_windows, models.fit_model, models.predict_closes)
display(Code("\n\n".join(inspect.getsource(function) for function in functions), language="python"))
lstm_5 = models.fit_model(train)
print("5 日 LSTM 训练完成。", flush=True)


5 日 LSTM 训练完成。


def complete_cutoffs(data, lookback=1, horizon=0):
    """Return cutoffs whose input and target rows are consecutive NYSE sessions."""
    dates = pd.DatetimeIndex(data.Date)
    if dates.empty or not dates.is_unique or not dates.is_monotonic_increasing:
        raise ValueError("Dates must be nonempty, unique and increasing.")
    calendar = xcals.get_calendar("XNYS", start=dates[0] - pd.Timedelta(days=14),
                                 end=dates[-1] + pd.Timedelta(days=14))
    positions = calendar.sessions_in_range(dates[0], dates[-1]).get_indexer(dates)
    if (positions < 0).any():
        raise ValueError("Data dates must be NYSE trading sessions.")
    cutoffs = np.arange(lookback - 1, len(data) - horizon)
    span = positions[cutoffs + horizon] - positions[cutoffs - lookback + 1]
    return cutoffs[span == lookback - 1 + horizon]


class ForecastModel(nn.Module):
    def __init__(self, horizon=SHORT_HORIZON, hidden_size=LSTM_SETTINGS["hidden_size"]):
        super().__init__()
        self.lstm = nn.LSTM(len(FEATURES), hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, inputs):
        sequence, _ = self.lstm(inputs)
        return self.output(sequence[:, -1])


def make_windows(features, cutoffs):
    return torch.from_numpy(np.stack([
        features[i - LOOKBACK + 1 : i + 1] for i in cutoffs
    ]))


def fit_model(train, horizon=SHORT_HORIZON):
    """Fit the LSTM using inputs and targets within the given history."""
    torch.manual_seed(LSTM_SETTINGS["seed"])
    torch.set_num_threads(2)
    train_cutoffs = complete_cutoffs(train, LOOKBACK, horizon)
    if not len(train_cutoffs):
        raise ValueError(f"Need a complete {LOOKBACK + horizon}-session training span.")
    features = train[FEATURES].to_numpy(dtype=np.float32)
    mean = features.mean(axis=0)
    std = np.maximum(features.std(axis=0), 1e-8)
    features = (features - mean) / std

    close = train.Close.to_numpy()
    days = np.arange(1, horizon + 1)
    changes = close[train_cutoffs[:, None] + days] - close[train_cutoffs, None]
    labels = torch.tensor(changes / POINT_SCALE, dtype=torch.float32)
    batches = DataLoader(TensorDataset(make_windows(features, train_cutoffs), labels),
                         batch_size=LSTM_SETTINGS["batch_size"])

    model = ForecastModel(horizon)
    optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_SETTINGS["learning_rate"])
    loss = nn.L1Loss()
    for _ in range(LSTM_SETTINGS["epochs"]):
        for inputs, targets in batches:
            optimizer.zero_grad()
            loss(model(inputs), targets).backward()
            optimizer.step()
    return model, mean, std


def predict_closes(model, mean, std, history, cutoffs):
    cutoffs = np.asarray(cutoffs, dtype=int)
    if not np.isin(cutoffs, complete_cutoffs(history, LOOKBACK)).all():
        raise ValueError(f"Each prediction requires {LOOKBACK} consecutive NYSE input sessions.")
    features = (history[FEATURES].to_numpy(dtype=np.float32) - mean) / std
    close = history.Close.to_numpy()
    model.eval()
    with torch.no_grad():
        changes = model(make_windows(features, cutoffs)).numpy() * POINT_SCALE
    return close[cutoffs, None] + changes

## 3. 验证与测试

按 NYSE 交易日历安排互不重叠的目标窗口，并要求截止日前有连续 30 个输入交易日。缺失日期不填充，输入或目标缺失的窗口均跳过。

5 日实验使用固定训练集：验证 226 个、测试 219 个窗口。20 日实验逐窗口重训：验证 56 个、测试 54 个窗口。
每次训练的输入、目标和标准化参数均限定在当次可用历史内。

In [4]:
results = {(horizon, split): evaluate(data, clean, horizon, split, lstm_5=lstm_5)
           for horizon in (5, 20) for split in ("val", "test")}


5 日 val：226 个窗口
5 日 test：219 个窗口
20 日 val：56 个窗口
  已完成 10/56 个窗口
  已完成 20/56 个窗口
  已完成 30/56 个窗口
  已完成 40/56 个窗口
  已完成 50/56 个窗口
  已完成 56/56 个窗口
20 日 test：54 个窗口
  已完成 10/54 个窗口
  已完成 20/54 个窗口
  已完成 30/54 个窗口
  已完成 40/54 个窗口
  已完成 50/54 个窗口
  已完成 54/54 个窗口


这里只展示整体 **MAE**，即所有窗口、全部步长的平均绝对误差，单位为指数点，越小越好。先看前两行测试集，在同一行比较三种方法。绿色单元格标出最低误差；Naive 是基线。逐步长的差异在下一部分的热力图里查看。

In [5]:
overall, chart_results = [], {5: {}, 20: {}}
for (horizon, split), (actual, predictions) in results.items():
    stage = f"{'Validation' if split == 'val' else 'Test'} ({len(actual)} windows)"
    chart_results[horizon][stage] = {
        name: metrics(actual, pred, axis=0)["mae"] for name, pred in predictions.items()
    }
    for name, pred in predictions.items():
        overall.append({"天数": horizon, "集合": split, "窗口数": len(actual), "模型": name,
                        **metrics(actual, pred, axis=None)})

scores = pd.DataFrame(overall)
scores["集合"] = scores["集合"].map({"val": "验证集", "test": "测试集"})
mae_table = scores.pivot(index=["天数", "集合"], columns="模型", values="mae")
mae_table = mae_table.reindex(index=[(5, "测试集"), (20, "测试集"), (5, "验证集"), (20, "验证集")],
                              columns=["Naive", "XGBoost", "LSTM"])
display(mae_table.style.format("{:.3f}").highlight_min(axis=1, color="#ddf2e7"))

,模型,Naive,XGBoost,LSTM
天数,集合,,,
5,测试集,68.311,68.005,67.736
20,测试集,119.560,117.624,116.968
5,验证集,45.052,44.798,44.461
20,验证集,74.808,72.946,72.262


## 4. 可视化

用本次运行的 MAE 展示整体改善比例柱状图和逐步长改善热力图。每张图下方有简短解释。

In [6]:
from io import BytesIO
from IPython.display import Image

import PlotForecastMAE as charts


def show_chart(fig):
    with BytesIO() as image:
        fig.savefig(image, format="png", dpi=120, bbox_inches="tight", pad_inches=0.12)
        display(Image(data=image.getvalue()))
    charts.plt.close(fig)


In [7]:
show_chart(charts.plot_overall_improvement(results=chart_results))

柱高表示相对 Naive 的整体 MAE 下降比例，越高越好。例如 2.50% 表示平均绝对误差比 Naive 低约 2.50%。左侧为验证集，右侧为测试集。

In [8]:
show_chart(charts.plot_step_improvement(results=chart_results))

每格是「Naive MAE − 模型 MAE」，单位为指数点。正值偏绿色，表示模型更准；负值表示 Naive 更准。上下两图各有自己的色标，跨图比较时读具体数值。

## 5. 课堂预测

修改下面三个日期后运行，会训练 LSTM、保存模型并导出预测 CSV。模型文件含权重、标准化参数、原 10 个特征、配置和训练截止日。默认保存到 src/1_TimeSerialAnalyze/models 和 reports/1_TimeSerialAnalyze/forecast_window.csv。复用已保存模型时，在 forecast_window 调用中增加 model_path=模型文件路径。

下面日期用于演示，正式提交时替换成老师公布的截止日和窗口。

In [9]:
from ClassroomForecast import DEFAULT_OUTPUT, forecast_window

CUTOFF, START, END = "2026-11-13", "2026-11-16", "2026-11-20"
forecast, model_path = forecast_window(CUTOFF, START, END)
display(forecast.rename(columns={"date": "交易日", "step": "步长", "predicted_close": "预测收盘点位"}))
print(f"模型已保存至 {model_path}\n预测 CSV 已保存至 {DEFAULT_OUTPUT}")


模型已保存至 D:\数据科学开源工具\sp500-multitask-main\src\1_TimeSerialAnalyze\models\lstm_20261113_5day.pt
预测 CSV 已保存至 D:\数据科学开源工具\sp500-multitask-main\reports\1_TimeSerialAnalyze\forecast_window.csv


按 2026-11-13 截止的历史训练 5 日 LSTM。


,交易日,步长,预测收盘点位
0,2026-11-16,1,7564.23
1,2026-11-17,2,7559.15
2,2026-11-18,3,7564.55
3,2026-11-19,4,7568.79
4,2026-11-20,5,7567.23
